# Gemma 4 Agent Lab — offline evaluation
Adapted from the [official starter](https://www.kaggle.com/code/ryanholbrook/getting-started-gemma-4-developer-agent).
This private notebook evaluates selected public tasks and emits submission.zip, task_results.jsonl, patches, and provenance. It does not submit to the leaderboard.


In [ ]:
import glob
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = "1"
os.environ["VLLM_ENGINE_READY_TIMEOUT_S"] = "1200"
os.environ["VLLM_NO_USAGE_STATS"] = "1"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

WHEELHOUSE_DIR = Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse")

if not any(WHEELHOUSE_DIR.glob("*.whl")):
    matches = [
        p
        for p in Path("/kaggle/input").rglob("gemma-4-developer-agent-wheelhouse")
        if p.is_dir() and any(p.glob("*.whl"))
    ]
    assert len(matches) == 1, (
        "Expected one mounted official wheelhouse",
        list(Path("/kaggle/input").iterdir()),
    )
    WHEELHOUSE_DIR = matches[0]

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    "/usr/local/lib/python*/dist-packages/*cutlass*.pth",
    "/usr/local/lib/python*/site-packages/*cutlass*.pth",
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path("/tmp/wheelhouse")
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob("*.whl"):
    if "cutlass" in w.name.lower():
        continue
    target_name = (
        w.name.replace("cu128", "+cu128") if ("cu128" in w.name and "+" not in w.name) else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob("*.whl"))
print(f"Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", *wheels],
    check=True,
)
importlib.invalidate_caches()
print("Wheelhouse installation complete.")

In [ ]:
import base64, hashlib, importlib.metadata, io, json, zipfile
from swegemma.models import load_tasks

DATA_DIR = Path("/kaggle/input/competitions/gemma-4-developer-agent")
WORKING_DIR = Path("/kaggle/working")
AGENT_DIR = WORKING_DIR / "candidate"
AGENT_DIR.mkdir(parents=True, exist_ok=False)
payload = base64.b64decode(
    "..."
)
assert (
    hashlib.sha256(payload).hexdigest()
    == "62937003fe519fa62e4c48f317abd6534d8ab80df25485e0929b18ede76c206d"
)
with zipfile.ZipFile(io.BytesIO(payload)) as zf:
    zf.extractall(AGENT_DIR)  # Locally validated, hash-pinned payload
DATA_WAIT_INTERVAL_SECONDS = 5
DATA_WAIT_TIMEOUT_SECONDS = 300
DATA_WAIT_LOG_EVERY_SECONDS = 30
DATA_PROBE_TIMEOUT_SECONDS = 30
DATA_PROBE_PATH = "/kaggle/working/data_mount_probe.txt"
DATA_READ_TIMEOUT_SECONDS = 10


def wait_for_tasks_jsonl(
    path,
    *,
    interval=DATA_WAIT_INTERVAL_SECONDS,
    timeout=DATA_WAIT_TIMEOUT_SECONDS,
    log_every=DATA_WAIT_LOG_EVERY_SECONDS,
    probe_timeout=DATA_PROBE_TIMEOUT_SECONDS,
    probe_path=DATA_PROBE_PATH,
    read_timeout=DATA_READ_TIMEOUT_SECONDS,
    clock=None,
    sleep=None,
    runner=None,
):
    """Poll until ``path`` looks like a finished tasks file.

    Blank and whitespace-only lines are ignored, and a final newline is not
    required, matching ``load_tasks``. Every other line must parse as a JSON
    object. The file is ready only when that is true and its size is unchanged
    across two polls, so a half-written file whose last line happens to parse
    is not accepted at once. A file that is already valid therefore waits one
    interval. The returned wait is rounded to one decimal.

    The first miss is logged with the reason (missing, empty, still changing,
    truncated, or unparseable), then again about every ``log_every`` seconds.
    After ``timeout``, print mount diagnostics and raise FileNotFoundError.
    ``stat`` and the read run on a daemon thread. If they are still going after
    ``read_timeout``, that poll counts as not ready. Any other exception does too.
    """
    import json
    import threading
    import time
    from pathlib import Path

    path = Path(path)
    if clock is None:
        clock = time.monotonic
    if sleep is None:
        sleep = time.sleep
    start = clock()
    deadline = start + timeout
    next_log = start
    logged = False
    reason = "missing"
    seen_size = None

    def classify_text(text):
        records = [line for line in text.splitlines() if line.strip()]
        if not records:
            return "empty"
        last = len(records) - 1
        for index, line in enumerate(records):
            try:
                value = json.loads(line)
            except Exception:
                return "truncated" if index == last else "unparseable"
            if not isinstance(value, dict):
                return "unparseable"
        return None

    def snapshot():
        # A hung mount must not block the poll. join returns and the thread is
        # a daemon, so this call gives up after read_timeout.
        box = {}

        def work():
            try:
                if not path.exists() or not path.is_file():
                    box["state"] = "missing"
                    return
                size = path.stat().st_size
                if size <= 0:
                    box["state"] = "empty"
                    return
                text = path.read_text(encoding="utf-8")
            except Exception:
                try:
                    if not path.exists():
                        box["state"] = "missing"
                        return
                except Exception:
                    box["state"] = "unreadable"
                    return
                box["state"] = "unreadable"
                return
            box["size"] = size
            box["state"] = classify_text(text)

        worker = threading.Thread(target=work, daemon=True)
        worker.start()
        worker.join(read_timeout)
        if worker.is_alive() or "state" not in box:
            return "unreadable", None
        return box["state"], box.get("size")

    while True:
        now = clock()
        try:
            state, size = snapshot()
        except Exception:
            state, size = "unreadable", None
        if state is None and size is not None and seen_size is not None and size == seen_size:
            waited = round(now - start, 1)
            print(f"tasks.jsonl ready after {waited:.1f}s: {path}", flush=True)
            return waited
        if state is None:
            reason = "still changing"
            seen_size = size
        else:
            reason = state
            seen_size = None
        if now >= deadline:
            break
        if not logged or now + 1e-9 >= next_log:
            elapsed = now - start
            print(f"waiting for {path} ({reason}, {elapsed:.0f}s elapsed)", flush=True)
            logged = True
            next_log = now + log_every
        remaining = deadline - now
        if remaining <= 0:
            break
        sleep(min(interval, remaining))

    def mount_probe():
        import subprocess

        commands = (
            ["ls", "-la", "/kaggle/input", "/kaggle/input/competitions"],
            ["find", "/kaggle/input", "-maxdepth", "4", "-name", "tasks.jsonl"],
        )
        chunks = []
        for cmd in commands:
            header = "$ " + " ".join(cmd) + "\n"
            try:
                if runner is None:
                    proc = subprocess.run(
                        cmd,
                        capture_output=True,
                        text=True,
                        timeout=probe_timeout,
                    )
                else:
                    proc = runner(cmd, probe_timeout)
                out = getattr(proc, "stdout", "") or ""
                err = getattr(proc, "stderr", "") or ""
                body = f"{out}{err}"
                if body and not body.endswith("\n"):
                    body += "\n"
                chunks.append(header + body)
            except Exception as exc:
                chunks.append(header + f"probe failed: {exc}\n")
        return "\n".join(chunks)

    waited = clock() - start
    text = mount_probe()
    print(text, end="" if text.endswith("\n") else "\n", flush=True)
    dest = Path(probe_path)
    try:
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_text(text)
    except OSError as exc:
        print(f"could not write probe to {dest}: {exc}", flush=True)
    raise FileNotFoundError(
        f"{path} was not ready after {waited:.0f}s (limit {timeout:g}s); "
        f"last check: {reason}. Probe output: {dest}"
    )


TASKS_PATH = DATA_DIR / "tasks.jsonl"
_data_wait_seconds = wait_for_tasks_jsonl(TASKS_PATH)
try:
    _run_manifest_path = WORKING_DIR / "run_manifest.json"
    _run_manifest = {}
    if _run_manifest_path.is_file():
        _loaded = json.loads(_run_manifest_path.read_text())
        if isinstance(_loaded, dict):
            _run_manifest = _loaded
    _run_manifest["data_wait_seconds"] = _data_wait_seconds
    _run_manifest_path.write_text(json.dumps(_run_manifest, indent=2) + "\n")
except (OSError, ValueError):
    pass
tasks = load_tasks(TASKS_PATH)
TASK_IDS = [
    "fastapi_11194",
    "fastapi_14246",
    "fastapi_14262",
    "fastapi_14356",
    "fastapi_14786",
    "fastapi_14794",
    "rich_3105",
    "rich_3454",
    "rich_3472",
    "rich_3934",
    "rich_3938",
    "rich_3944",
    "rich_4070",
]
if TASK_IDS is None:
    TASK_IDS = [t.instance_id for t in tasks[:2]]
assert set(TASK_IDS) <= {t.instance_id for t in tasks}, "Unknown task IDs"
RUN_PROVENANCE = {
    "created_at": "2026-10-21T06:14:54.877389+00:00",
    "git_revision": "bd85a6aac9339ade47c2335b6d5e19870e137e2b",
    "packing_commit": "bd85a6aac9339ade47c2335b6d5e19870e137e2b",
    "worktree_dirty": False,
    "candidate_source_commit": "bd85a6aac9339ade47c2335b6d5e19870e137e2b",
    "candidate_source_dirty": False,
    "sha256": "62937003fe519fa62e4c48f317abd6534d8ab80df25485e0929b18ede76c206d",
    "source": "/workspace/agents/structured-v16-2",
    "archive": "/tmp/tmpb9e69c4r/submission.zip",
    "files": {
        "agent.yaml": "02c43d26901d338911883c26b5a4a77739e8bbe9a99bb6acfccde4ae3cb7bd23",
        "eval_config.yaml": "bdfe547f5a4a737bb811633ed75974fec377137e2dcb6b39c12973838cc1f47f",
        "skills/check/SKILL.md": "53be3c89f61d5fb21bb1b6026f62e0e07af9ef41eff7a6ea23a10dc4e036e933",
        "skills/check/scripts/run.py": "d8fcfccdc7b7e3be73f10aeeb2811faef1fc5d7015c1bb4b28223dd16b9eae9c",
        "sub_agents/repair_1.md": "1d2f0a08f53fb7788e6deea0ef3535e726afc969025e7d72e7401ae1d067dce7",
        "sub_agents/repair_1.yaml": "b69f62f4104613c455f2b38c71c5bcb96fe2e1a5c8c9328fa7f33d3fcf62722a",
        "sub_agents/repair_2.md": "78d103cd6cbd309ecdb5c4ad717b3c0f5c673d318d1448f2bf234476de17e064",
        "sub_agents/repair_2.yaml": "af481480c6e7c660e2f5e3af58e0582b25ee789c5ddc1fc8150c91f385f93dc3",
        "sub_agents/repair_3.md": "f3a8e8b0ad6cc06a17ecad319663cd18a20328f344aefc2f0272ae54745533b3",
        "sub_agents/repair_3.yaml": "3a0dd19fa4ee9fbc1dcd16ba8de55f0068029e5a83c290f6ed28b1a64136bb96",
        "sub_agents/repair_thinking.yaml": "e8e34853dd3a81eae86424ca85f49399de32b0e7905137fc0ce456242c697edf",
        "sub_agents/thinking.yaml": "c065a5fc63a9a371a6a890064d1d6d43fc35dc68714c8b233ca3c84e79b0173c",
        "sub_agents/triage.md": "6468d531d87712f8afcb8a9eb20873f39d4874a39393a28ff4e6f2cf30dc2b80",
        "sub_agents/triage.yaml": "ee8657dc2e664d639ecabf35580485ba0a0bb6325f723eb2bec4f2a2bc73ea6e",
        "sub_agents/verify_1.md": "8c363cb0c759ac57104faaa7ad82735cd35f7f1f164355ea3ce0720f5626fff3",
        "sub_agents/verify_1.yaml": "473f80a31d44ae41ce93c70675968778822ca90c8cdeddf94e3fb5ba0db3ed4d",
        "sub_agents/verify_2.md": "e088b9db7b324effb79d5e265a4c49afb18d647ead0f79a4d5058d48c3c40fb3",
        "sub_agents/verify_2.yaml": "af497f5b94edd47e68657395f3650a0b384ede597e5a81043386fd3db0badce7",
        "sub_agents/verify_3.md": "611c91d0eb8b661b284081f43f8bc3254858a14b22d178a89b76e1f3562e0a28",
        "sub_agents/verify_3.yaml": "11de71f05e91c10ea3e0510c7f0dd323cc4dc9e217d6158632a61cf633eb6372",
    },
    "validation": "portable checks only; official GPU harness required",
}
RUN_PROVENANCE["task_ids"] = TASK_IDS
RUN_PROVENANCE["packages"] = {
    name: importlib.metadata.version(name)
    for name in ["swegemma", "adk-submission", "adk-eval-core", "vllm", "google-adk"]
}
RUN_PROVENANCE["task_file_sha256"] = hashlib.sha256(TASKS_PATH.read_bytes()).hexdigest()
RUN_PROVENANCE["data_wait_seconds"] = _data_wait_seconds
(WORKING_DIR / "run_manifest.json").write_text(json.dumps(RUN_PROVENANCE, indent=2))
print(f"Loaded {len(tasks)} tasks; evaluating {len(TASK_IDS)} selected tasks")

In [ ]:
from swegemma import graph as sg

sample_task = tasks[0]
GRAPH_DIR = str(DATA_DIR / "graphs")
EMBEDDINGS_DIR = str(DATA_DIR / "embeddings")

# Load the pre-computed repository code graph and node embeddings for the sample task
repo_graph = sg.get_graph(
    repo_name=sample_task.repo,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
sample_nodes = list(repo_graph.nodes())
print(f"Graph for {sample_task.repo} ({sample_task.instance_id}):")
print(f"  Nodes: {repo_graph.number_of_nodes()}, Edges: {repo_graph.number_of_edges()}")

# Select a connected symbol and query structural neighbors (callers, callees, definitions)
query_node = next((n for n, deg in repo_graph.degree() if deg >= 5), sample_nodes[0])
neighbors = sg.get_neighbor(
    node=query_node,
    graph=repo_graph,
    max_neighbors=10,
)
print(f"\nNeighbors of {query_node!r} (showing up to 5):")
for n in neighbors[:5]:
    print(f"  - {n}")

# Search for semantically similar code nodes using pre-computed vector embeddings
similar = sg.get_similar_nodes(
    node=query_node,
    repo_name=sample_task.repo,
    k=5,
    graph=repo_graph,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
print(f"\nTop similar nodes to {query_node!r}:")
for item in similar:
    print(f"  - {item['node_name']} (similarity={item['similarity']:.4f})")

# Extract an induced subgraph connecting the query symbol and its neighbors
focal_nodes = [query_node, *neighbors[:4]]
subgraph = sg.get_induced_subgraph(repo_graph, focal_nodes)
print(
    f"\nInduced subgraph over {len(focal_nodes)} focal nodes: "
    f"{subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges"
)

In [ ]:
import time as _gemma_lab_time

_gemma_lab_model_load_started = _gemma_lab_time.perf_counter()
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

try:
    from swegemma.models.discovery import validate_single_declared_model
except (ImportError, ModuleNotFoundError):
    from adk_submission.discovery import discover_declared_models

    def _norm(name):
        v = name.strip().lower()
        for p in ("openai/", "google/", "hosted_vllm/", "custom/"):
            if v.startswith(p):
                v = v[len(p) :]
        return v

    def validate_single_declared_model(agent_dir):
        models = discover_declared_models(agent_dir, normalize_fn=_norm)
        if not models:
            raise ValueError("No model declared in agent configuration.")
        if len(models) > 1:
            raise ValueError(f"Ambiguous model declaration: {sorted(models)}")
        return next(iter(models))


litellm.drop_params = True

TARGET_MODEL_NAME = "gemma-4-31b-it-qat-w4a16-ct"
MODEL_PATH = Path("/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2")
INFERENCE_API_KEY = "EMPTY"

# Validate single base model and discover any PEFT LoRA adapters in the submission directory
declared_model = validate_single_declared_model(AGENT_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host="127.0.0.1",
    tool_call_parser="gemma4",
    reasoning_parser="gemma4",
    default_chat_template_kwargs={"enable_thinking": True},
    max_model_len=32768,
    dtype="bfloat16" if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else "auto",
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=True,
    max_loras=8,
    max_lora_rank=128,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f"vLLM server started on {server_instance.base_url} (tp={tp_size})")

# Register model and LoRA adapter aliases for Google ADK
models = server_instance.create_model_registry(
    aliases=[declared_model, TARGET_MODEL_NAME],
    model_prefix="openai/",
    api_key=INFERENCE_API_KEY,
)
RUN_PROVENANCE["model_load_seconds"] = (
    _gemma_lab_time.perf_counter() - _gemma_lab_model_load_started
)
RUN_PROVENANCE["hardware"] = {
    "gpu_count": torch.cuda.device_count(),
    "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "tensor_parallel_size": tp_size,
}
(WORKING_DIR / "run_manifest.json").write_text(json.dumps(RUN_PROVENANCE, indent=2))

In [ ]:
import asyncio
import concurrent.futures
import pandas as pd
import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator


def run_sync(coro_or_fn, *args, **kwargs):
    """Run an async coroutine synchronously inside a notebook event loop."""
    fn = (lambda: coro_or_fn(*args, **kwargs)) if callable(coro_or_fn) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())


# Read evaluation settings from the submission's eval_config.yaml
eval_config_file = AGENT_DIR / "eval_config.yaml"
raw_eval_cfg = yaml.safe_load(eval_config_file.read_text(encoding="utf-8"))
eval_section = raw_eval_cfg.get("evaluation", raw_eval_cfg)

# Set your own budget constraints here
timeout_seconds = int(eval_section.get("timeout_seconds", 300))
max_tool_calls = 220
max_time_minutes = 4.0
turns_raw = eval_section.get("max_turns", eval_section.get("max_llm_calls"))
max_turns = int(turns_raw) if turns_raw is not None else None

# Select a small subset of tasks from the published dataset
SAMPLE_TASKS = [t for t in tasks if t.instance_id in TASK_IDS]
limits, gen_constraints = build_submission_limits()

# Configure Evaluator to run Phase 1 (agent patch generation) and Phase 2 (verification)
# using the subprocess sandbox backend and the submission's eval_config parameters
eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / "snapshots",
    results_dir=WORKING_DIR / "results",
    submission_dir=AGENT_DIR,
    models=models,
    sandbox="subprocess",
    timeout_seconds=timeout_seconds,
    max_time_minutes=max_time_minutes,
    max_tool_calls=max_tool_calls,
    max_turns=max_turns,
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15,
        overlap_size=2,
        token_threshold=14336,
        event_retention_size=5,
    ),
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    wheels_dir=DATA_DIR / "wheels",
    verbose=False,
)

evaluator = Evaluator(eval_config)
predictions = []

results_path = WORKING_DIR / "task_results.jsonl"
for idx, task in enumerate(SAMPLE_TASKS, start=1):
    print(f"[{idx}/{len(SAMPLE_TASKS)}] {task.instance_id}")
    try:
        result = run_sync(
            evaluator.evaluate_task, task=task, task_index=idx, total_tasks=len(SAMPLE_TASKS)
        )
        patch = result.agent_patch or ""
        row = {
            "instance_id": task.instance_id,
            "repo": task.repo,
            "resolved": bool(result.resolved),
            "test_exit_code": result.test_exit_code,
            "patch_chars": len(patch),
            "tool_calls": result.tool_calls,
            "duration_seconds": result.duration_seconds,
        }
        # The evaluator can return status SUCCESS while recording a runner failure.
        runner_error = getattr(result, "error_message", None)
        if runner_error:
            row["agent_error"] = runner_error
            if any(
                marker in runner_error.lower()
                for marker in ["exceeded session timeout", "exceeded turns budget"]
            ):
                row["failure_class"] = "agent_budget"
            else:
                row["failure_class"] = "infrastructure_or_harness"
                row["error"] = runner_error
        if (
            "recursive dependency involving fixture 'httpbin'" in (result.test_output or "")
            and "b/tests/conftest.py" not in patch
        ):
            row["failure_class"] = "infrastructure_or_harness"
            row["error"] = "Official verification httpbin fixture setup failed"
        (WORKING_DIR / "results" / "patches").mkdir(parents=True, exist_ok=True)
        (WORKING_DIR / "results" / "patches" / f"{task.instance_id}.patch").write_text(patch)
        # Persist the harness result, including available trace and verification diagnostics.
        if hasattr(result, "model_dump"):
            details = result.model_dump()
        elif hasattr(result, "__dict__"):
            details = vars(result)
        else:
            details = {"result": str(result)}

        def _jsonable(value, _seen=None):
            if isinstance(value, (str, int, float, bool)) or value is None:
                return value
            if _seen is None:
                _seen = set()
            identity = id(value)
            if identity in _seen:
                return None
            if isinstance(value, dict):
                _seen.add(identity)
                return {str(key): _jsonable(item, _seen) for key, item in value.items()}
            if isinstance(value, (list, tuple)):
                _seen.add(identity)
                return [_jsonable(item, _seen) for item in value]
            dump = getattr(value, "to_dict", None)
            if not callable(dump):
                dump = getattr(value, "model_dump", None)
            if callable(dump):
                _seen.add(identity)
                try:
                    return _jsonable(dump(), _seen)
                except Exception:
                    pass
            if hasattr(value, "__dict__"):
                _seen.add(identity)
                return _jsonable(vars(value), _seen)
            return str(value)

        # Evaluator record only. The harness writes ATIF to
        # results/traces/trace_<id>.json; this cell must not replace that file.
        (WORKING_DIR / "results" / f"{task.instance_id}.json").write_text(
            json.dumps(_jsonable(details), indent=2)
        )
    except Exception as exc:
        import traceback

        traceback.print_exc()
        patch = ""
        row = {
            "instance_id": task.instance_id,
            "repo": task.repo,
            "resolved": False,
            "error": repr(exc),
            "failure_class": "infrastructure_or_harness",
        }
    with results_path.open("a") as handle:
        handle.write(json.dumps(row) + "\n")
    predictions.append({"id": task.instance_id, "prediction": patch})
    print(row)

submission_df = pd.DataFrame(predictions, columns=["id", "prediction"])
display(submission_df)

In [ ]:
import zipfile
from swegemma.config import ALLOWED_SUBMISSION_EXTENSIONS, MAX_SUBMISSION_SIZE_BYTES

# Create submission.zip with agent.yaml at the root of the archive
zip_base = WORKING_DIR / "submission"
zip_path = WORKING_DIR / "submission.zip"
zip_path.write_bytes(payload)

# Validate archive contents against competition constraints
with zipfile.ZipFile(zip_path, "r") as zf:
    infos = zf.infolist()
    total_size = sum(i.file_size for i in infos)
    assert total_size <= MAX_SUBMISSION_SIZE_BYTES, f"Archive exceeds 3 GiB limit: {total_size}"
    for info in infos:
        if not info.is_dir():
            ext = Path(info.filename).suffix.lower()
            assert ext in ALLOWED_SUBMISSION_EXTENSIONS, (
                f"Disallowed file extension: {info.filename}"
            )

print(f"Created {zip_path} ({zip_path.stat().st_size / (1024 * 1024):.2f} MiB)")

In [ ]:
server_instance.stop()

## After the official loop

Point `V162_RESULTS` at the directory that holds `task_results.jsonl` and
`results/traces/`. This cell does not start the model and does not delete traces.


In [ ]:
import json
import os
import re
from pathlib import Path

EDIT_MARKERS = ("line_fix", "cat >", "cat>>", "tee ")
LATER_ROLES = {"verify_1", "repair_2", "verify_2", "repair_3", "verify_3"}
CAP_SECONDS = 240
_ELAPSED_MARK = re.compile(r"\[elapsed (\d+)s\]")


def _events(trace):
    if isinstance(trace, dict):
        if isinstance(trace.get("steps"), list):
            return trace["steps"]
        return trace.get("events") or trace.get("trace") or []
    return []


def _author(event):
    if not isinstance(event, dict):
        return ""
    extra = event.get("extra") if isinstance(event.get("extra"), dict) else {}
    for source in (event, extra):
        for key in ("author", "agent", "agent_name", "role"):
            value = source.get(key)
            if value:
                return str(value)
    return ""


def _elapsed(event):
    if not isinstance(event, dict):
        return None
    extra = event.get("extra") if isinstance(event.get("extra"), dict) else {}
    for source in (event, extra):
        for key in (
            "elapsed_s",
            "agent_elapsed_seconds",
            "elapsed_seconds",
            "elapsed",
        ):
            value = source.get(key)
            if value is None:
                continue
            try:
                return float(value)
            except (TypeError, ValueError):
                continue
    return None


def _tool_name(event):
    if not isinstance(event, dict):
        return ""
    return str(event.get("tool") or event.get("name") or event.get("tool_name") or "")


def _blob(event):
    return json.dumps(event, default=str)


def _is_edit(event):
    name = _tool_name(event)
    if name in {"edit_file", "write_file"}:
        return True
    return any(marker in _blob(event) for marker in EDIT_MARKERS)


def _hypothesis(event):
    return "HYPOTHESIS:" in _blob(event)


def _malformed_repair_1(event):
    """Truncated markup or unparseable arguments on a repair_1 tool call."""
    if _author(event) != "repair_1":
        return False
    blob = _blob(event)
    if "tool_call|>" in blob:
        return True
    if blob.count("<tool_call") > blob.count("</tool_call>"):
        return True
    for key in ("args", "arguments"):
        value = event.get(key)
        if isinstance(value, str):
            try:
                json.loads(value)
            except json.JSONDecodeError:
                return True
    return False


def hypothesis_elapsed(trace):
    """First [elapsed] mark at or before the first HYPOTHESIS line."""
    last = None
    for event in _events(trace):
        blob = _blob(event)
        index = blob.find("HYPOTHESIS:")
        if index >= 0:
            marks = [int(match.group(1)) for match in _ELAPSED_MARK.finditer(blob[:index])]
            if marks:
                return marks[-1]
            return last
        marks = [int(match.group(1)) for match in _ELAPSED_MARK.finditer(blob)]
        if marks:
            last = marks[-1]
    return None


def repair_1_handoff(trace, wall):
    """Whether repair_1 returned before the cap, and the hand-off elapsed."""
    handoff = None
    returned = False
    last_repair = None
    for event in _events(trace):
        author = _author(event)
        elapsed = _elapsed(event)
        if author == "repair_1" and elapsed is not None:
            last_repair = elapsed
        if author in LATER_ROLES:
            returned = True
            handoff = last_repair if last_repair is not None else elapsed
            break
    if not returned:
        return {"returned_before_cap": False, "handoff_elapsed": None}
    before_cap = handoff is None or handoff < CAP_SECONDS
    if wall is not None and handoff is None:
        try:
            before_cap = float(wall) < CAP_SECONDS
        except (TypeError, ValueError):
            before_cap = True
    return {"returned_before_cap": before_cap, "handoff_elapsed": handoff}


def summarize(rows, traces):
    tasks = []
    for row in rows:
        task_id = row.get("instance_id") or row.get("task_id") or ""
        trace = traces.get(task_id) or {}
        events = _events(trace)
        calls = 0
        first = None
        edited = False
        hypothesis = False
        handoff_due = False
        end_round_1 = False
        malformed = 0
        for event in events:
            blob = _blob(event)
            if _tool_name(event):
                calls += 1
            if "HYPOTHESIS:" in blob:
                hypothesis = True
            if "HAND-OFF DUE" in blob:
                handoff_due = True
            if "END_ROUND_1" in blob:
                end_round_1 = True
            if _malformed_repair_1(event):
                malformed += 1
            if first is None and _is_edit(event):
                first = calls
                edited = True
        if not edited:
            patch = str(row.get("patch") or row.get("model_patch") or "")
            edited = bool(patch.strip())
        wall = row.get("wall_seconds") or row.get("duration_seconds")
        handoff = repair_1_handoff(trace, wall)
        tasks.append(
            {
                "instance_id": task_id,
                "resolved": bool(row.get("resolved") or row.get("passed")),
                "wall_seconds": wall,
                "no_edit": not edited,
                "calls_to_first_edit": first,
                "hypothesis": hypothesis,
                "hypothesis_elapsed": hypothesis_elapsed(trace),
                "handoff_due": handoff_due,
                "end_round_1": end_round_1,
                "malformed_repair_1_calls": malformed,
                "repair_1_returned_before_cap": handoff["returned_before_cap"],
                "handoff_elapsed": handoff["handoff_elapsed"],
            }
        )
    return {
        "n": len(tasks),
        "resolved": sum(item["resolved"] for item in tasks),
        "no_edit_runs": sum(item["no_edit"] for item in tasks),
        "hypothesis_runs": sum(item["hypothesis"] for item in tasks),
        "handoff_due_runs": sum(item["handoff_due"] for item in tasks),
        "end_round_1_runs": sum(item["end_round_1"] for item in tasks),
        "malformed_repair_1_calls": sum(item["malformed_repair_1_calls"] for item in tasks),
        "repair_1_returned_before_cap": sum(item["repair_1_returned_before_cap"] for item in tasks),
        "tasks": tasks,
    }


RESULTS = Path(os.environ.get("V162_RESULTS", "/kaggle/working"))
rows = []
results_path = RESULTS / "task_results.jsonl"
if results_path.is_file():
    for line in results_path.read_text().splitlines():
        if line.strip():
            rows.append(json.loads(line))
traces = {}
trace_dir = RESULTS / "results" / "traces"
if not trace_dir.is_dir():
    trace_dir = RESULTS / "traces"
if trace_dir.is_dir():
    for path in sorted(trace_dir.glob("*.json")):
        payload = json.loads(path.read_text())
        task_id = payload.get("instance_id") or path.stem.removeprefix("trace_")
        traces[task_id] = payload
report = summarize(rows, traces)
print("resolved", report["resolved"], "of", report["n"])
print("no_edit_runs", report["no_edit_runs"])
print("hypothesis_runs", report["hypothesis_runs"])
print("handoff_due_runs", report["handoff_due_runs"])
print("end_round_1_runs", report["end_round_1_runs"])
print("malformed_repair_1_calls", report["malformed_repair_1_calls"])
print("repair_1_returned_before_cap", report["repair_1_returned_before_cap"])
for item in report["tasks"]:
    if item["hypothesis_elapsed"] is None:
        mark = "no_[elapsed]"
    else:
        mark = f"[elapsed {item['hypothesis_elapsed']}s]"
    print(
        item["instance_id"],
        "resolved" if item["resolved"] else "unresolved",
        "wall",
        item["wall_seconds"],
        "no_edit" if item["no_edit"] else "edit",
        "calls_to_first_edit",
        item["calls_to_first_edit"],
        "HYPOTHESIS" if item["hypothesis"] else "no_HYPOTHESIS",
        "hypothesis_elapsed",
        mark,
        "HAND-OFF DUE" if item["handoff_due"] else "no_HAND-OFF_DUE",
        "END_ROUND_1" if item["end_round_1"] else "no_END_ROUND_1",
        "malformed_repair_1_calls",
        item["malformed_repair_1_calls"],
        (
            "repair_1_returned_before_cap"
            if item["repair_1_returned_before_cap"]
            else "repair_1_held_cap"
        ),
        "handoff_elapsed",
        item["handoff_elapsed"],
    )